# Recovered ZuCo: diagnostics + simple classification
Select **CPU**, then **Run all**, using Zyion (account 2) and the same Drive mount as the completed reproduction.

This reads saved raw/clean trials once. **No native EEG download, cleaning or extraction.** The small original SST text ZIP may be read in the cloud for label verification. All EEG/features stay on runtime disk; only small outputs/checkpoints are saved to a **separate folder**, capped at **50 MB total including the later CNN**.

Order: (1) diagnostics and linear decoding run together; (2) categorized results below; (3) optional separate T4 CNN notebook. Completed fits resume without replacing the old paper figures.


## Connect the same Drive account

In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
RUN=THESIS/'Results/word_sentiment_eeg/zuco_paper_figures_recovered'
OUT=THESIS/'Results/word_sentiment_eeg/zuco_recovered_followups'
REPO=Path('/content/word-sentiment-eeg-recovered-followups')
CODE_REVISION='6a4624e40725740e8bde7d8e993023477267b5ff'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
print('Quota account:',response.json().get('user',{}).get('emailAddress','unavailable'))
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)

used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES < max(0,50_000_000-used):
    raise RuntimeError('Need up to 50 MB free for all new follow-up outputs; existing results unchanged.')


## Run diagnostics and fixed classification experiments
Three group splits: unseen sentences, unseen word spellings, unseen readers. Fixed class-weighted logistic model plus class-prior dummy baseline; raw/clean EEG; 224–304-ms mean and 40-ms bins over 0–800 ms; pre-onset control; individual and disjoint 20/40-trial averages. A metadata-only model checks word/timing separability without EEG.

Diagnostics include exact reader sign-flip window tests (Holm over four tests), SST lookup flags, covariate balance, reader correlation matrices, original/recovered-block comparisons and repeated-word similarity. Source archives and completed paper results are read-only. Classification progress prints each case; an interrupted run skips completed cases after reloading saved trials. This can take tens of minutes depending on Drive/CPU speed; it does not perform the multi-hour native extraction.


In [ ]:
from datetime import datetime, timezone
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/zuco_recovered_followups.py'),
         '--thesis-root',str(THESIS),'--phase','cpu']
# Advisory status stays on runtime disk; scientific progress/results persist on Drive.
run_with_progress(command,Path('/content/zuco-followups-status-cpu.json'),activity='Recovered ZuCo cpu follow-ups on saved trials')


## 1. Additional published-window tests

In [ ]:
tests=json.loads((OUT/'diagnostics/window_tests.json').read_text())
display(pd.DataFrame(tests['tests']))
print(tests['warning'])
print('These are additional equal-reader whole-map tests, not the existing trial-paired TCT or corrected clusters.')


## 2. Labels and matching covariates

In [ ]:
audit=json.loads((OUT/'diagnostics/label_audit.json').read_text())
labels=pd.DataFrame(audit['rows'])
flags=labels[labels.matches_direct_dictionary.ne(True)]
print('Accepted word/label/method combinations:',len(labels),'  Dictionary review flags:',len(flags))
display(flags.head(30))
print('Full word audit:',OUT/'diagnostics/label_audit.json')
for line in audit['limitations']:print(line)
balance=pd.DataFrame(json.loads((OUT/'diagnostics/covariate_balance.json').read_text()))
display(balance[balance.reader.eq('ALL_TRIALS')])
print('Per-reader balance and mutually exclusive selection counts are also saved in diagnostics/.')


## 3. Reader and recovered-block consistency

In [ ]:
display(Image(filename=str(OUT/'figures/reader_map_correlations.png')))
display(Image(filename=str(OUT/'figures/controls_and_repeats.png')))
display(pd.DataFrame(json.loads((OUT/'diagnostics/block_set_counts.json').read_text())))
print(json.loads((OUT/'diagnostics/block_set_consistency.json').read_text())['warning'])


## 4. Single-fixation classification — first look

In [ ]:
summary=json.loads((OUT/'classification/summary.json').read_text())
rows=[]
for s in summary:
    p=s.get('pooled') or {}
    rows.append({k:v for k,v in s.items() if k!='pooled'}|{k:p.get(k) for k in ['n','accuracy','balanced_accuracy','macro_f1','auc']})
scores=pd.DataFrame(rows)
display(scores[(scores.average_trials.eq(1)) & scores.window.isin(['published_224_304','post_0_800'])].sort_values(['split','window','representation','model']))
print('Binary balanced-accuracy chance = 0.50. Scores are exploratory; many cases are displayed, not selected as a single confirmed finding.')


## 5. Averages of 20/40 trials — comparison with the paper

In [ ]:
display(scores[scores.average_trials.gt(1)].sort_values(['split','window','representation','average_trials','model']))
print('Class-known disjoint averages formed inside reader/class AFTER splitting. Remainders excluded; insufficient folds flagged.')
print('The paper reports approximately 60% accuracy on 40-trial averages. Our grouped splits, metric and classifier differ; this is not exact decoding replication.')


## 6. Pre-onset and metadata-only controls

In [ ]:
display(scores[scores.window.isin(['pre_minus200_0','metadata_only'])])
print('Metadata: word length, saved corpus frequency, fixation duration, previous/next gaps. Scaling and imputation are fitted on training data only.')
print('Each case JSON includes train/test group overlaps, actual averaging counts, reader/fold scores and confusion matrices.')


## 7. Repeated words

In [ ]:
repeat=json.loads((OUT/'diagnostics/repeated_words.json').read_text())
display(pd.DataFrame(repeat['rows']))
print(repeat['warning'])
print('All new outputs:',OUT)
print('Saved follow-up footprint (MB):',round(sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file())/1e6,2))


## Interpretation
All results are exploratory. The original cleaning and matching used full reader recordings, so these are **transductive preprocessing** evaluations even when classifier folds hold out sentences, words or readers. They cannot establish strict fit-on-training-only preprocessing generalization. Shared words/sentences across other split types are reported. Pre-onset EEG is a context/timing control, not guaranteed sentiment-free data.

Pseudotrial accuracy uses known labels to build same-class test averages and is **not single-fixation decoding accuracy**. No neutral trials are present in this binary archive. No settings or labels are changed to make a result significant.


## Next: secondary CNN on T4 GPU
[Open the separate CNN notebook](https://colab.research.google.com/github/parmisbathayan/word-sentiment-eeg/blob/main/notebooks/zuco_recovered_followups_cnn.ipynb?authuser=2).

The authors’ email draft and full fixed plan are in the GitHub docs folder. Email has **not** been sent. No further native extraction is required for either notebook.
